In [1]:
import pandas as pd
import numpy as np

reduced_dataset = pd.read_csv(
    "../data/processed/keyboard_reduced_productivity_dataset.csv"
)

print("Shape:", reduced_dataset.shape)
print("\nColumns:")
print(reduced_dataset.columns.tolist())

Shape: (10000, 10)

Columns:
['participant', 'session', 'window_id', 'DU.key1.key1_median', 'DU.key1.key2_median', 'unique_key1_count', 'unique_transition_count', 'consecutive_key1_repetition_rate', 'backspace_rate', 'productivity_score']


In [2]:
# Selected behavioral features
reduced_features = [
    "DU.key1.key1_median",
    "DU.key1.key2_median",
    "unique_key1_count",
    "unique_transition_count",
    "consecutive_key1_repetition_rate",
    "backspace_rate"
]

# Input features
X = reduced_dataset[reduced_features]

# Target
y = reduced_dataset["productivity_score"]

# Participant-session groups
groups = (
    reduced_dataset["participant"].astype(str)
    + "_session_"
    + reduced_dataset["session"].astype(str)
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", len(reduced_features))
print("Unique participant-session groups:", groups.nunique())

X shape: (10000, 6)
y shape: (10000,)
Number of features: 6
Unique participant-session groups: 197


In [3]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining groups:", groups_train.nunique())
print("Testing groups:", groups_test.nunique())

overlap = set(groups_train).intersection(set(groups_test))

print("\nOverlapping participant-session groups:", len(overlap))

Training samples: 7971
Testing samples: 2029

Training groups: 157
Testing groups: 40

Overlapping participant-session groups: 0


In [4]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Calculate metrics
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print("---------------------")
print("MAE :", mae_rf)
print("MSE :", mse_rf)
print("RMSE:", rmse_rf)
print("R²  :", r2_rf)

Random Forest Results
---------------------
MAE : 1.5499294922245475
MSE : 4.390979279656605
RMSE: 2.0954663632844612
R²  : 0.9801748558670504


In [6]:
rf_importance = pd.DataFrame({
    "feature": reduced_features,
    "importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("Random Forest Feature Importance")
print("---------------------------------")
print(rf_importance)

Random Forest Feature Importance
---------------------------------
                            feature  importance
0  consecutive_key1_repetition_rate    0.412063
1               DU.key1.key1_median    0.159486
2                 unique_key1_count    0.149125
3                    backspace_rate    0.109670
4               DU.key1.key2_median    0.091081
5           unique_transition_count    0.078575


In [7]:
rf_results = pd.DataFrame({
    "model": ["Random Forest"],
    "MAE": [mae_rf],
    "MSE": [mse_rf],
    "RMSE": [rmse_rf],
    "R2": [r2_rf]
})

rf_results.to_csv(
    "../data/processed/random_forest_results.csv",
    index=False
)

rf_importance.to_csv(
    "../data/processed/random_forest_feature_importance.csv",
    index=False
)

print("Random Forest results saved successfully.")

Random Forest results saved successfully.
